## 🔐 Крок 1: Підключення до Azure OpenAI

Перший крок – ініціалізувати клієнт `ChatCompletionsClient`, вказавши endpoint та ключ доступу.

In [1]:
import os
import json
import requests
from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import ChatCompletionsToolDefinition
from azure.core.credentials import AzureKeyCredential

load_dotenv()
token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.inference.ai.azure.com"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

# Виберіть модель загального призначення для тексту
deployment = "gpt-4o"


## 🔧 Крок 2: Визначення функції

Функцію описуємо у форматі JSON Schema (OpenAI Function Calling format) і передаємо як `ChatCompletionsToolDefinition`.

> У нашому випадку, функція `search_courses` — імітує пошук курсів з документації Microsoft Learn за заданими критеріями.

### Елементи функціонального виклику

Ось пояснення ключових частин функціонального виклику:

- **name** - Ім'я функції, яку ми хочемо викликати
- **description** - Опис того, як працює функція. Важливо бути конкретним і чітким
- **parameters** - Список значень та форматів, які модель повинна використовувати у відповіді
- **type** - Тип даних властивостей
- **properties** - Список конкретних значень, які модель буде використовувати для відповіді
- **required** - Обов'язкові властивості для виконання функціонального виклику

In [ ]:
# 🔧 Визначення функції, яку модель може викликати
functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_courses",
            "description": "Returns a list of training courses from the Microsoft catalog",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": """User role (for example: developer, student)"""
                    },
                    "product": {
                        "type": "string",
                        "description": "Covered product (Azure, Power BI, etc.)"
                    },
                    "level": {
                        "type": "string",
                        "description": "User experience level"
                    }
                },
                "required": ["role"]
            }
        }
    )
]


## 📤 Крок 3: Надсилання повідомлення користувача

Ми надсилаємо запит від імені користувача (наприклад, "знайди курси для початківця-розробника по Azure"), і модель вирішує, чи слід викликати функцію.

Щоб інтегрувати функцію у виклик Chat Completion API, ми додаємо `tools=functions` до запиту. Встановлення `tool_choice="auto"` дозволяє LLM самостійно вирішити, яку функцію викликати, виходячи з повідомлення користувача.

In [3]:
# 🧠 Створення повідомлення та виклик моделі з інструментами
messages = [
    {
        "role": "user",
        "content": "Find a course for a beginner developer on Azure"
    }
]

response = client.complete(
    model=deployment,
    messages=messages,
    tools=functions,
    tool_choice="auto"
)

response_message = response.choices[0].message
print("📥 Відповідь моделі:")
print(response_message)


📥 Відповідь моделі:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"role":"developer","product":"Azure","level":"beginner"}', 'name': 'search_courses'}, 'id': 'call_n2vZ5UssUFiU7Rmyxx8b5Wm3', 'type': 'function'}]}


## ⚙️ Крок 4: Обробка виклику функції (tool_call)

Якщо модель вирішила викликати функцію, ми зчитуємо її аргументи, викликаємо локальну функцію `search_courses`, і передаємо результат назад в модель як `"role": "tool"`.

Після цього модель формує підсумкову відповідь з урахуванням виклику.

### Інтеграція у застосунок

Для інтеграції у реальний застосунок необхідно:
1. Перевірити, чи модель хоче викликати функцію
2. Отримати аргументи з відповіді моделі
3. Викликати відповідну функцію з отриманими аргументами
4. Додати відповідь від функції до історії повідомлень
5. Зробити новий запит до моделі для отримання підсумкової відповіді користувачу

In [4]:
# ⚙️ Обробка function_call (tool_call)
tool_calls = response_message.tool_calls

if tool_calls and len(tool_calls) > 0:
    first_tool_call = tool_calls[0]
    function_name = first_tool_call.function.name
    function_args = json.loads(first_tool_call.function.arguments)

    def search_courses(role, product, level):
        url = "https://learn.microsoft.com/api/catalog/"
        params = {
            "role": role,
            "product": product,
            "level": level
        }
        response = requests.get(url, params=params)
        modules = response.json().get("modules", [])
        results = []
        for module in modules[:5]:
            title = module.get("title")
            url = module.get("url")
            results.append({"title": title, "url": url})
        return json.dumps(results, ensure_ascii=False)
        

    available_functions = {
        "search_courses": search_courses,
    }

    function_to_call = available_functions[function_name]
    function_response = function_to_call(**function_args)

    print("✅ Результат виклику функції:")
    print(function_response)

    # Додаємо відповідь моделі з tool_calls до історії повідомлень
    messages.append({
        "role": "assistant",
        "content": "Відповідай українською",
        "tool_calls": [
            {
                "id": first_tool_call.id,
                "type": "function",
                "function": {
                    "name": first_tool_call.function.name,
                    "arguments": first_tool_call.function.arguments
                }
            }
        ]
    })

    # Додаємо відповідь функції
    messages.append({
        "role": "tool",  # У найновіших версіях API використовується "tool" замість "function"
        "tool_call_id": first_tool_call.id,  # Це критично!
        "name": function_name,
        "content": function_response
    })

    # Отримуємо фінальну відповідь від моделі
    next_response = client.complete(
        model=deployment,
        messages=messages
    )

    print("📤 Остаточна відповідь моделі:")
    print(next_response.choices[0].message.content)
else:
    print("⚠️ Модель не захотіла викликати функцію.")


✅ Результат виклику функції:
[{"title": "Guide AI workload operations with an AI Center of Excellence", "url": "https://learn.microsoft.com/en-us/training/modules/guide-ai-operations-center-excellence/?WT.mc_id=api_CatalogApi"}, {"title": "Develop products with screen reader support", "url": "https://learn.microsoft.com/en-us/training/modules/develop-products-with-screen-reader-support/?WT.mc_id=api_CatalogApi"}, {"title": "Host a web application with Azure App Service", "url": "https://learn.microsoft.com/en-us/training/modules/host-a-web-app-with-azure-app-service/?WT.mc_id=api_CatalogApi"}, {"title": "Deploy to multiple Azure environments by using JSON ARM template features", "url": "https://learn.microsoft.com/en-us/training/modules/modify-azure-resource-manager-template-reuse/?WT.mc_id=api_CatalogApi"}, {"title": "Secure Azure OpenAI authentication and authorization", "url": "https://learn.microsoft.com/en-us/training/modules/secure-azure-openai-authentication-authorization/?WT.mc

## Висновки та практичні завдання

Тепер ви знаєте, як інтегрувати Function Calling у ваші застосунки з Azure OpenAI. Ця потужна функціональність дозволяє створювати більш інтелектуальні та корисні взаємодії з користувачами.

### Завдання для подальшого вивчення:

1. Додайте більше параметрів до функції, які можуть допомогти учням знаходити більше курсів. Перегляньте доступні API параметри [тут](https://learn.microsoft.com/training/support/catalog-api-developer-reference).
2. Створіть інший функціональний виклик, який отримує додаткову інформацію від користувача, наприклад, його рідну мову.
3. Додайте обробку помилок для випадків, коли функціональний виклик та/або API-виклик не повертає відповідних курсів.
4. Розширте функціонал для рекомендації не лише курсів, але й навчальних шляхів (learning paths).

___

1. Додайте більше параметрів до функції, які можуть допомогти учням знаходити більше курсів. Перегляньте доступні API параметри [тут](https://learn.microsoft.com/training/support/catalog-api-developer-reference).

In [2]:
functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_courses",
            "description": "Returns a list of training courses from the Microsoft catalog",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": "User role (for example: developer, student, administrator)"
                    },
                    "product": {
                        "type": "string",
                        "description": "Covered product (Azure, Power BI, Microsoft 365, Windows, etc.)"
                    },
                    "level": {
                        "type": "string",
                        "description": "User experience level (beginner, intermediate, advanced)"
                    },
                    "type": {
                        "type": "string",
                        "description": "Type of learning content (module, learning path, certification)"
                    },
                    "duration": {
                        "type": "string",
                        "description": "Estimated duration in minutes or hours"
                    },
                    "language": {
                        "type": "string",
                        "description": "Content language (en-us, fr-fr, de-de, etc.)"
                    },
                    "skills": {
                        "type": "string",
                        "description": "Specific skills or technologies (AI, machine learning, security, etc.)"
                    },
                    "provider": {
                        "type": "string",
                        "description": "Content provider (Microsoft Learn, GitHub, etc.)"
                    },
                    "format": {
                        "type": "string",
                        "description": "Learning format (interactive, video, documentation)"
                    },
                    "last_updated": {
                        "type": "string",
                        "description": "When the content was last updated"
                    },
                    "popularity": {
                        "type": "string",
                        "description": "Popularity or rating of the content"
                    },
                    "free": {
                        "type": "boolean",
                        "description": "Whether the content is free"
                    },
                    "certification": {
                        "type": "boolean",
                        "description": "Whether the content leads to certification"
                    },
                    "interactive": {
                        "type": "boolean",
                        "description": "Whether the content includes interactive elements"
                    }
                },
                "required": ["role"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "get_user_preferences",
            "description": "Get additional user information and preferences to provide better course recommendations",
            "parameters": {
                "type": "object",
                "properties": {
                    "native_language": {
                        "type": "string",
                        "description": "User's native language (Ukrainian, English, French, German, Spanish, etc.)"
                    },
                    "preferred_language": {
                        "type": "string",
                        "description": "Preferred language for learning content"
                    },
                    "learning_goals": {
                        "type": "string",
                        "description": "User's specific learning objectives and goals"
                    },
                    "time_commitment": {
                        "type": "string",
                        "description": "Available time for learning per week"
                    },
                    "preferred_format": {
                        "type": "string",
                        "description": "Preferred learning format (video, text, interactive, hands-on)"
                    },
                    "prior_experience": {
                        "type": "string",
                        "description": "User's prior experience with the topic"
                    },
                    "deadline": {
                        "type": "string",
                        "description": "Learning deadline or timeline"
                    },
                    "accessibility_needs": {
                        "type": "string",
                        "description": "Any accessibility requirements"
                    }
                },
                "required": ["native_language", "preferred_language"]
            }
        }
    )
]

2. Створіть інший функціональний виклик, який отримує додаткову інформацію від користувача, наприклад, його рідну мову.

In [3]:
messages = [
    {
        "role": "user",
        "content": "Find a course for a beginner developer on Azure"
    }
]

response = client.complete(
    model=deployment,
    messages=messages,
    tools=functions,
    tool_choice="auto"
)

response_message = response.choices[0].message
print("📥 Відповідь моделі:")
print(response_message)

📥 Відповідь моделі:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"role":"developer","product":"Azure","level":"beginner"}', 'name': 'search_courses'}, 'id': 'call_NTbxnPEe2MQJoGR2BeMXbZgO', 'type': 'function'}]}


3. Додайте обробку помилок для випадків, коли функціональний виклик та/або API-виклик не повертає відповідних курсів.

In [4]:
# ⚙️ Обробка function_call (tool_call)
tool_calls = response_message.tool_calls

if tool_calls and len(tool_calls) > 0:
    first_tool_call = tool_calls[0]
    function_name = first_tool_call.function.name
    function_args = json.loads(first_tool_call.function.arguments)

    def search_courses(role, product=None, level=None):
        try:
            url = "https://learn.microsoft.com/api/catalog/"
            params = {"role": role}
            
            if product:
                params["product"] = product
            if level:
                params["level"] = level
            
            print(f"🔍 Виконується пошук курсів з параметрами: {params}")
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            
            modules = response.json().get("modules", [])
            
            if not modules:
                error_message = "❌ Не знайдено курсів за вказаними критеріями. Спробуйте змінити параметри пошуку."
                return json.dumps({"error": error_message, "courses_found": 0}, ensure_ascii=False)
            
            results = []
            for module in modules[:5]:
                title = module.get("title", "Без назви")
                url = module.get("url", "#")
                # Формуємо повну URL-адресу
                full_url = f"https://learn.microsoft.com{url}" if url.startswith('/') else url
                results.append({
                    "title": title, 
                    "url": full_url,
                    "summary": module.get("summary", "Опис відсутній")
                })
            
            return json.dumps({
                "success": True,
                "courses_found": len(results),
                "courses": results
            }, ensure_ascii=False)
            
        except requests.exceptions.RequestException as e:
            error_message = f"❌ Помилка при з'єднанні з API: {str(e)}"
            return json.dumps({"error": error_message, "courses_found": 0}, ensure_ascii=False)
        except json.JSONDecodeError as e:
            error_message = f"❌ Помилка обробки відповіді: {str(e)}"
            return json.dumps({"error": error_message, "courses_found": 0}, ensure_ascii=False)
        except Exception as e:
            error_message = f"❌ Неочікувана помилка: {str(e)}"
            return json.dumps({"error": error_message, "courses_found": 0}, ensure_ascii=False)

    available_functions = {
        "search_courses": search_courses,
    }

    if function_name in available_functions:
        function_to_call = available_functions[function_name]
        function_response = function_to_call(**function_args)

        print("✅ Результат виклику функції:")
        print(function_response)

        # Додаємо відповідь моделі з tool_calls до історії повідомлень
        messages.append({
            "role": "assistant",
            "content": "Відповідай українською",
            "tool_calls": [
                {
                    "id": first_tool_call.id,
                    "type": "function",
                    "function": {
                        "name": first_tool_call.function.name,
                        "arguments": first_tool_call.function.arguments
                    }
                }
            ]
        })

        # Додаємо відповідь функції
        messages.append({
            "role": "tool",
            "tool_call_id": first_tool_call.id,
            "name": function_name,
            "content": function_response
        })

        try:
            # Отримуємо фінальну відповідь від моделі
            next_response = client.complete(
                model=deployment,
                messages=messages
            )

            print("📤 Остаточна відповідь моделі:")
            print(next_response.choices[0].message.content)
            
        except Exception as e:
            print(f"❌ Помилка при отриманні відповіді від моделі: {e}")
            print("💡 Спробуйте виконати запит ще раз.")
            
    else:
        print(f"⚠️ Функція '{function_name}' не підтримується.")
        # Додаємо повідомлення про помилку для моделі
        messages.append({
            "role": "tool",
            "tool_call_id": first_tool_call.id,
            "name": function_name,
            "content": json.dumps({"error": f"Функція '{function_name}' не знайдена"})
        })
        
        # Запитуємо модель про помилку
        try:
            error_response = client.complete(
                model=deployment,
                messages=messages
            )
            print("📤 Відповідь моделі на помилку:")
            print(error_response.choices[0].message.content)
        except Exception as e:
            print(f"❌ Помилка: {e}")
            
else:
    print("⚠️ Модель не захотіла викликати функцію.")

🔍 Виконується пошук курсів з параметрами: {'role': 'developer', 'product': 'Azure', 'level': 'beginner'}
✅ Результат виклику функції:
{"success": true, "courses_found": 5, "courses": [{"title": "Guide AI workload operations with an AI Center of Excellence", "url": "https://learn.microsoft.com/en-us/training/modules/guide-ai-operations-center-excellence/?WT.mc_id=api_CatalogApi", "summary": "How comprehensive operations guidance from an AI Center of Excellence can help an organization to effectively deploy, manage, and maintain AI workloads."}, {"title": "Develop products with screen reader support", "url": "https://learn.microsoft.com/en-us/training/modules/develop-products-with-screen-reader-support/?WT.mc_id=api_CatalogApi", "summary": "Learn how to develop products with screen reader support to ensure accessibility for users who are blind or have low vision. This module covers essential principles, design considerations, and practical tips for creating inclusive websites and apps th

4. Розширте функціонал для рекомендації не лише курсів, але й навчальних шляхів (learning paths).

In [5]:
# ⚙️ Обробка function_call (tool_call)
tool_calls = response_message.tool_calls

if tool_calls and len(tool_calls) > 0:
    first_tool_call = tool_calls[0]
    function_name = first_tool_call.function.name
    function_args = json.loads(first_tool_call.function.arguments)

    def search_courses(role, product=None, level=None, content_type=None, skills=None):
        try:
            url = "https://learn.microsoft.com/api/catalog/"
            params = {"role": role}
            
            if product:
                params["product"] = product
            if level:
                params["level"] = level
            if skills:
                params["skills"] = skills
            
            print(f"🔍 Виконується пошук контенту з параметрами: {params}")
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            
            data = response.json()
            modules = data.get("modules", [])
            learning_paths = data.get("learningPaths", [])
            
            # Фільтруємо за типом контенту, якщо вказано
            if content_type == "course" or content_type == "module":
                learning_paths = []
            elif content_type == "learning_path":
                modules = []
            
            results = {
                "courses": [],
                "learning_paths": []
            }
            
            # Обробка курсів/модулів
            for module in modules[:5]:
                title = module.get("title", "Без назви")
                url = module.get("url", "#")
                full_url = f"https://learn.microsoft.com{url}" if url.startswith('/') else url
                results["courses"].append({
                    "type": "course",
                    "title": title,
                    "url": full_url,
                    "summary": module.get("summary", "Опис відсутній"),
                    "duration": module.get("duration", "Невідомо"),
                    "level": module.get("level", "Не вказано")
                })
            
            # Обробка навчальних шляхів
            for path in learning_paths[:5]:
                title = path.get("title", "Без назви")
                url = path.get("url", "#")
                full_url = f"https://learn.microsoft.com{url}" if url.startswith('/') else url
                results["learning_paths"].append({
                    "type": "learning_path",
                    "title": title,
                    "url": full_url,
                    "summary": path.get("summary", "Опис відсутній"),
                    "modules_count": path.get("modules_count", 0),
                    "duration": path.get("duration", "Невідомо")
                })
            
            total_found = len(results["courses"]) + len(results["learning_paths"])
            
            if total_found == 0:
                error_message = "❌ Не знайдено контенту за вказаними критеріями. Спробуйте змінити параметри пошуку."
                return json.dumps({"error": error_message, "total_found": 0}, ensure_ascii=False)
            
            return json.dumps({
                "success": True,
                "total_found": total_found,
                "courses_found": len(results["courses"]),
                "learning_paths_found": len(results["learning_paths"]),
                "content": results
            }, ensure_ascii=False)
            
        except requests.exceptions.RequestException as e:
            error_message = f"❌ Помилка при з'єднанні з API: {str(e)}"
            return json.dumps({"error": error_message, "total_found": 0}, ensure_ascii=False)
        except json.JSONDecodeError as e:
            error_message = f"❌ Помилка обробки відповіді: {str(e)}"
            return json.dumps({"error": error_message, "total_found": 0}, ensure_ascii=False)
        except Exception as e:
            error_message = f"❌ Неочікувана помилка: {str(e)}"
            return json.dumps({"error": error_message, "total_found": 0}, ensure_ascii=False)

    def recommend_learning_content(role, preferred_content_type="both", product=None, level=None, skills=None):
        """
        Розширена функція для рекомендації курсів та навчальних шляхів
        """
        try:
            # Визначаємо тип контенту для пошуку
            content_type_map = {
                "courses_only": "course",
                "paths_only": "learning_path", 
                "both": None
            }
            
            content_type = content_type_map.get(preferred_content_type)
            
            # Використовуємо основну функцію пошуку
            search_result = search_courses(role, product, level, content_type, skills)
            result_data = json.loads(search_result)
            
            if "error" in result_data:
                return search_result
            
            # Додаємо рекомендації на основі знайденого контенту
            content = result_data.get("content", {})
            courses = content.get("courses", [])
            learning_paths = content.get("learning_paths", [])
            
            recommendations = {
                "for_beginners": [],
                "comprehensive_learning": [],
                "quick_start": []
            }
            
            # Рекомендації для початківців
            beginner_courses = [c for c in courses if "fundamental" in c.get("title", "").lower() or "beginner" in c.get("level", "").lower()]
            recommendations["for_beginners"] = beginner_courses[:2]
            
            # Комплексні навчальні шляхи
            if learning_paths:
                recommendations["comprehensive_learning"] = learning_paths[:2]
            elif courses:
                # Якщо шляхів немає, рекомендуємо послідовність курсів
                recommendations["comprehensive_learning"] = courses[:3]
            
            # Швидкий старт - короткі курси
            short_courses = [c for c in courses if "min" in c.get("duration", "").lower()]
            recommendations["quick_start"] = short_courses[:2] if short_courses else courses[:2]
            
            result_data["recommendations"] = recommendations
            
            return json.dumps(result_data, ensure_ascii=False)
            
        except Exception as e:
            error_message = f"❌ Помилка при формуванні рекомендацій: {str(e)}"
            return json.dumps({"error": error_message, "total_found": 0}, ensure_ascii=False)

    available_functions = {
        "search_courses": search_courses,
        "recommend_learning_content": recommend_learning_content,
    }

    if function_name in available_functions:
        function_to_call = available_functions[function_name]
        function_response = function_to_call(**function_args)

        print("✅ Результат виклику функції:")
        print(function_response)

        # Додаємо відповідь моделі з tool_calls до історії повідомлень
        messages.append({
            "role": "assistant",
            "content": "Відповідай українською мовою, надай детальні рекомендації щодо курсів та навчальних шляхів",
            "tool_calls": [
                {
                    "id": first_tool_call.id,
                    "type": "function",
                    "function": {
                        "name": first_tool_call.function.name,
                        "arguments": first_tool_call.function.arguments
                    }
                }
            ]
        })

        # Додаємо відповідь функції
        messages.append({
            "role": "tool",
            "tool_call_id": first_tool_call.id,
            "name": function_name,
            "content": function_response
        })

        try:
            # Отримуємо фінальну відповідь від моделі
            next_response = client.complete(
                model=deployment,
                messages=messages
            )

            print("📤 Остаточна відповідь моделі:")
            print(next_response.choices[0].message.content)
            
        except Exception as e:
            print(f"❌ Помилка при отриманні відповіді від моделі: {e}")
            print("💡 Спробуйте виконати запит ще раз.")
            
    else:
        print(f"⚠️ Функція '{function_name}' не підтримується.")
        messages.append({
            "role": "tool",
            "tool_call_id": first_tool_call.id,
            "name": function_name,
            "content": json.dumps({"error": f"Функція '{function_name}' не знайдена"})
        })
        
        try:
            error_response = client.complete(
                model=deployment,
                messages=messages
            )
            print("📤 Відповідь моделі на помилку:")
            print(error_response.choices[0].message.content)
        except Exception as e:
            print(f"❌ Помилка: {e}")
            
else:
    print("⚠️ Модель не захотіла викликати функцію.")

🔍 Виконується пошук контенту з параметрами: {'role': 'developer', 'product': 'Azure', 'level': 'beginner'}
✅ Результат виклику функції:
{"success": true, "total_found": 10, "courses_found": 5, "learning_paths_found": 5, "content": {"courses": [{"type": "course", "title": "Guide AI workload operations with an AI Center of Excellence", "url": "https://learn.microsoft.com/en-us/training/modules/guide-ai-operations-center-excellence/?WT.mc_id=api_CatalogApi", "summary": "How comprehensive operations guidance from an AI Center of Excellence can help an organization to effectively deploy, manage, and maintain AI workloads.", "duration": "Невідомо", "level": "Не вказано"}, {"type": "course", "title": "Develop products with screen reader support", "url": "https://learn.microsoft.com/en-us/training/modules/develop-products-with-screen-reader-support/?WT.mc_id=api_CatalogApi", "summary": "Learn how to develop products with screen reader support to ensure accessibility for users who are blind or 

___

## Індивиідуальне завдання

### Частина 1: Базовий пошук курсів за тематикою
Модифікуйте базовий ноутбук, щоб реалізувати пошук курсів за іншою тематикою (згідно з вашим варіантом). 
1. Змініть опис функції search_courses, замінивши параметр product на subject:

In [6]:
def search_courses(role, subject, level=None):
    """
    Search for courses by subject area in Microsoft Learn catalog
    """
    try:
        url = "https://learn.microsoft.com/api/catalog/"
        params = {
            "role": role,
            "subject": subject
        }
        
        if level:
            params["level"] = level
            
        print(f"🔍 Searching courses for {role} in {subject}...")
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        modules = data.get("modules", [])
        
        if not modules:
            return json.dumps({
                "error": f"No courses found for {role} in {subject}",
                "suggestion": "Try different subject or role"
            }, ensure_ascii=False)
        
        results = []
        for module in modules[:5]:
            title = module.get("title", "No title")
            url = module.get("url", "#")
            # Ensure full URL
            full_url = f"https://learn.microsoft.com{url}" if url.startswith('/') else url
            results.append({
                "title": title, 
                "url": full_url,
                "summary": module.get("summary", "No description available"),
                "duration": module.get("duration", "Unknown")
            })
        
        return json.dumps({
            "success": True,
            "subject": subject,
            "role": role,
            "courses_found": len(results),
            "courses": results
        }, ensure_ascii=False)
        
    except requests.exceptions.RequestException as e:
        return json.dumps({
            "error": f"Network error: {str(e)}",
            "courses_found": 0
        }, ensure_ascii=False)
    except Exception as e:
        return json.dumps({
            "error": f"Unexpected error: {str(e)}",
            "courses_found": 0
        }, ensure_ascii=False)

2. Змініть функцію search_courses, щоб вона використовувала параметр subject замість product:

In [7]:
def search_courses(role, subject, level=None):
    """
    Search for courses by subject area in Microsoft Learn catalog
    """
    try:
        url = "https://learn.microsoft.com/api/catalog/"
        params = {
            "role": role,
            "subject": subject  # Змінено з product на subject
        }
        
        if level:
            params["level"] = level
            
        print(f"🔍 Пошук курсів для {role} з тематики {subject}...")
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        modules = data.get("modules", [])
        
        if not modules:
            return json.dumps({
                "error": f"Не знайдено курсів для {role} з тематики {subject}",
                "suggestion": "Спробуйте іншу тематику або роль"
            }, ensure_ascii=False)
        
        results = []
        for module in modules[:5]:
            title = module.get("title", "Без назви")
            url = module.get("url", "#")
            # Формуємо повну URL-адресу
            full_url = f"https://learn.microsoft.com{url}" if url.startswith('/') else url
            results.append({
                "title": title, 
                "url": full_url,
                "summary": module.get("summary", "Опис відсутній"),
                "duration": module.get("duration", "Невідомо"),
                "level": module.get("level", "Не вказано")
            })
        
        return json.dumps({
            "success": True,
            "subject": subject,  # Змінено з product на subject
            "role": role,
            "courses_found": len(results),
            "courses": results
        }, ensure_ascii=False)
        
    except requests.exceptions.RequestException as e:
        return json.dumps({
            "error": f"Помилка мережі: {str(e)}",
            "courses_found": 0
        }, ensure_ascii=False)
    except Exception as e:
        return json.dumps({
            "error": f"Неочікувана помилка: {str(e)}",
            "courses_found": 0
        }, ensure_ascii=False)

3. Змініть повідомлення користувача, щоб воно відповідало вашій тематиці:

In [9]:
# Для кібербезпеки
messages = [
    {
        "role": "user", 
        "content": "Потрібні курси з Кібербезпеки для початківців у IT"
    }
]

# Для хмарних технологій
messages = [
    {
        "role": "user",
        "content": "Знайди навчальні матеріали з Хмарних Обчислень для розробників"
    }
]

# Для веб-розробки
messages = [
    {
        "role": "user",
        "content": "Потрібні курси з Веб-розробки для студентів, рівень - початківець"
    }
]

# Для Data Science
messages = [
    {
        "role": "user", 
        "content": "Рекомендуй курси з Аналізу Даних для аналітиків середнього рівня"
    }
]

In [10]:
# 🧠 Створення повідомлення та виклик моделі з інструментами
messages = [
    {
        "role": "user",
        "content": "Знайди курси з Штучного Інтелекту для початківців у сфері Data Science"
    }
]

response = client.complete(
    model=deployment,
    messages=messages,
    tools=functions,
    tool_choice="auto"
)

response_message = response.choices[0].message
print("📥 Відповідь моделі:")
print(response_message)

📥 Відповідь моделі:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"role":"student","level":"beginner","skills":"Artificial Intelligence","product":"Data Science"}', 'name': 'search_courses'}, 'id': 'call_dRGnqT39hNkEuNEV5W95Ucjw', 'type': 'function'}]}


### Частина 2: Розширене завдання
Розширте функціональність базового рішення одним із наступних способів:

А) Додайте розширену фільтрацію результатів Модифікуйте функцію пошуку курсів, щоб вона підтримувала додаткові параметри фільтрації та аналізу:

In [17]:
def search_courses(role, subject, level=None, duration_filter=None, content_type=None, language="en-us", sort_by="popularity", max_results=10, free_only=False, certification_ready=False):
    """
    Розширена функція пошуку курсів з додатковою фільтрацією та аналізом
    """
    try:
        url = "https://learn.microsoft.com/api/catalog/"
        params = {
            "role": role,
            "subject": subject,
            "locale": language
        }
        
        if level:
            params["level"] = level
            
        print(f"🔍 Розширений пошук курсів для {role} з тематики {subject}...")
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        modules = data.get("modules", [])
        learning_paths = data.get("learningPaths", [])
        
        # Об'єднуємо модулі та навчальні шляхи
        all_content = modules + learning_paths
        
        if not all_content:
            return json.dumps({
                "error": f"Не знайдено контенту для {role} з тематики {subject}",
                "suggestion": "Спробуйте іншу тематику або роль"
            }, ensure_ascii=False)
        
        # Фільтрація за типом контенту
        if content_type == "course":
            filtered_content = modules
        elif content_type == "learning_path":
            filtered_content = learning_paths
        else:
            filtered_content = all_content
        
        results = []
        for item in filtered_content:
            # Базова інформація
            title = item.get("title", "Без назви")
            url = item.get("url", "#")
            full_url = f"https://learn.microsoft.com{url}" if url.startswith('/') else url
            item_type = "learning_path" if "learningPath" in str(item.get("type", "")) else "course"
            
            # Аналіз тривалості
            duration = item.get("duration", "Невідомо")
            duration_minutes = 0
            if "min" in duration.lower():
                duration_minutes = int(''.join(filter(str.isdigit, duration)))
            elif "hour" in duration.lower():
                duration_minutes = int(''.join(filter(str.isdigit, duration))) * 60
            
            # Фільтрація за тривалістю
            if duration_filter:
                if duration_filter == "short" and duration_minutes > 30:
                    continue
                elif duration_filter == "medium" and (duration_minutes <= 30 or duration_minutes > 120):
                    continue
                elif duration_filter == "long" and duration_minutes <= 120:
                    continue
            
            # Перевірка на безкоштовність (більшість контенту Microsoft Learn безкоштовний)
            is_free = True  # Припускаємо, що контент безкоштовний
            
            # Перевірка на підготовку до сертифікації
            is_certification_ready = "certification" in title.lower() or "exam" in title.lower() or "az-" in title.lower()
            
            if certification_ready and not is_certification_ready:
                continue
            
            # Аналіз популярності (симуляція)
            popularity_score = item.get("popularity", 0) or len(title) % 10  # Симуляція популярності
            
            course_data = {
                "title": title, 
                "url": full_url,
                "summary": item.get("summary", "Опис відсутній"),
                "duration": duration,
                "duration_minutes": duration_minutes,
                "level": item.get("level", "Не вказано"),
                "type": item_type,
                "popularity_score": popularity_score,
                "is_free": is_free,
                "is_certification_ready": is_certification_ready,
                "last_updated": item.get("last_updated", "Не вказано"),
                "skills": item.get("skills", []),
                "rating": item.get("rating", "Не вказано")
            }
            results.append(course_data)
        
        # Сортування результатів
        if sort_by == "popularity":
            results.sort(key=lambda x: x["popularity_score"], reverse=True)
        elif sort_by == "duration":
            results.sort(key=lambda x: x["duration_minutes"])
        elif sort_by == "title":
            results.sort(key=lambda x: x["title"])
        elif sort_by == "level":
            level_order = {"beginner": 1, "intermediate": 2, "advanced": 3}
            results.sort(key=lambda x: level_order.get(x["level"].lower(), 4))
        
        # Обмеження кількості результатів
        results = results[:max_results]
        
        # Аналітичні метрики
        total_courses = len(results)
        total_duration = sum(item["duration_minutes"] for item in results if item["duration_minutes"] > 0)
        certification_courses = sum(1 for item in results if item["is_certification_ready"])
        free_courses = sum(1 for item in results if item["is_free"])
        
        # Розподіл за рівнями
        level_distribution = {}
        for item in results:
            level = item["level"].lower()
            level_distribution[level] = level_distribution.get(level, 0) + 1
        
        return json.dumps({
            "success": True,
            "subject": subject,
            "role": role,
            "filters_applied": {
                "level": level,
                "duration_filter": duration_filter,
                "content_type": content_type,
                "language": language,
                "sort_by": sort_by,
                "max_results": max_results,
                "free_only": free_only,
                "certification_ready": certification_ready
            },
            "analytics": {
                "total_courses_found": total_courses,
                "total_estimated_duration_minutes": total_duration,
                "certification_ready_courses": certification_courses,
                "free_courses": free_courses,
                "level_distribution": level_distribution,
                "content_type_distribution": {
                    "courses": sum(1 for item in results if item["type"] == "course"),
                    "learning_paths": sum(1 for item in results if item["type"] == "learning_path")
                }
            },
            "courses": results
        }, ensure_ascii=False)
        
    except requests.exceptions.RequestException as e:
        return json.dumps({
            "error": f"Помилка мережі: {str(e)}",
            "courses_found": 0
        }, ensure_ascii=False)
    except Exception as e:
        return json.dumps({
            "error": f"Неочікувана помилка: {str(e)}",
            "courses_found": 0
        }, ensure_ascii=False)

# Оновлення опису функції в tools
functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_courses",
            "description": "Returns a list of training courses from the Microsoft catalog with advanced filtering and analytics",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": "User role (for example: developer, student, administrator, data scientist)"
                    },
                    "subject": {
                        "type": "string",
                        "description": "Subject area (Artificial Intelligence, Data Science, Cloud Computing, Cybersecurity, Web Development, etc.)"
                    },
                    "level": {
                        "type": "string",
                        "description": "User experience level (beginner, intermediate, advanced)"
                    },
                    "duration_filter": {
                        "type": "string",
                        "description": "Filter by duration (short: <30min, medium: 30min-2h, long: >2h)"
                    },
                    "content_type": {
                        "type": "string",
                        "description": "Type of content (course, learning_path, or both)"
                    },
                    "language": {
                        "type": "string",
                        "description": "Content language (en-us, uk-ua, fr-fr, de-de, etc.)"
                    },
                    "sort_by": {
                        "type": "string",
                        "description": "Sort results by (popularity, duration, title, level)"
                    },
                    "max_results": {
                        "type": "number",
                        "description": "Maximum number of results to return (default: 10)"
                    },
                    "free_only": {
                        "type": "boolean",
                        "description": "Show only free courses"
                    },
                    "certification_ready": {
                        "type": "boolean",
                        "description": "Show only courses that prepare for certification"
                    }
                },
                "required": ["role", "subject"]
            }
        }
    )
]

In [18]:
# Пошук коротких курсів з AI для сертифікації
search_courses(
    role="developer",
    subject="Artificial Intelligence",
    level="beginner",
    duration_filter="short",
    certification_ready=True,
    max_results=5,
    sort_by="popularity"
)

🔍 Розширений пошук курсів для developer з тематики Artificial Intelligence...


'{"error": "Не знайдено контенту для developer з тематики Artificial Intelligence", "suggestion": "Спробуйте іншу тематику або роль"}'

Б) Додайте інший функціональний виклик Додайте другу функцію, яка може бути викликана моделлю для отримання детальної інформації про курс:

In [20]:
def get_user_preferences(native_language, preferred_language, learning_goals=None, time_commitment=None, preferred_format=None):
    """
    Функція для збору інформації про користувача та його вподобання
    """
    try:
        user_data = {
            "native_language": native_language,
            "preferred_language": preferred_language,
            "learning_goals": learning_goals,
            "time_commitment": time_commitment,
            "preferred_format": preferred_format
        }
        
        return json.dumps({
            "success": True,
            "user_preferences": user_data,
            "message": "Інформація про користувача успішно збережена"
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка при обробці інформації користувача: {str(e)}"
        }, ensure_ascii=False)

In [21]:
# 🔧 Додаємо другу функцію для отримання детальної інформації про курс
functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_courses",
            "description": "Returns a list of training courses from the Microsoft catalog with advanced filtering and analytics",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": "User role (for example: developer, student, administrator, data scientist)"
                    },
                    "subject": {
                        "type": "string",
                        "description": "Subject area (Artificial Intelligence, Data Science, Cloud Computing, Cybersecurity, Web Development, etc.)"
                    },
                    "level": {
                        "type": "string",
                        "description": "User experience level (beginner, intermediate, advanced)"
                    },
                    "duration_filter": {
                        "type": "string",
                        "description": "Filter by duration (short: <30min, medium: 30min-2h, long: >2h)"
                    },
                    "content_type": {
                        "type": "string",
                        "description": "Type of content (course, learning_path, or both)"
                    },
                    "language": {
                        "type": "string",
                        "description": "Content language (en-us, uk-ua, fr-fr, de-de, etc.)"
                    },
                    "sort_by": {
                        "type": "string",
                        "description": "Sort results by (popularity, duration, title, level)"
                    },
                    "max_results": {
                        "type": "number",
                        "description": "Maximum number of results to return (default: 10)"
                    },
                    "free_only": {
                        "type": "boolean",
                        "description": "Show only free courses"
                    },
                    "certification_ready": {
                        "type": "boolean",
                        "description": "Show only courses that prepare for certification"
                    }
                },
                "required": ["role", "subject"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "get_course_details",
            "description": "Get detailed information about a specific course including syllabus, prerequisites, and learning objectives",
            "parameters": {
                "type": "object",
                "properties": {
                    "course_url": {
                        "type": "string",
                        "description": "URL of the course to get detailed information about"
                    },
                    "include_syllabus": {
                        "type": "boolean",
                        "description": "Whether to include detailed syllabus and modules"
                    },
                    "include_prerequisites": {
                        "type": "boolean",
                        "description": "Whether to include prerequisites and requirements"
                    },
                    "include_learning_objectives": {
                        "type": "boolean",
                        "description": "Whether to include learning objectives and outcomes"
                    },
                    "include_reviews": {
                        "type": "boolean",
                        "description": "Whether to include student reviews and ratings"
                    }
                },
                "required": ["course_url"]
            }
        }
    )
]

def get_course_details(course_url, include_syllabus=True, include_prerequisites=True, 
                      include_learning_objectives=True, include_reviews=False):
    """
    Отримання детальної інформації про конкретний курс
    """
    try:
        print(f"🔍 Отримання деталей курсу: {course_url}")
        
        # Для демонстрації симулюємо отримання даних з API
        # У реальному сценарії тут був би запит до API Microsoft Learn
        
        # Симуляція даних курсу на основі URL
        course_data = {
            "title": "Детальна інформація про курс",
            "url": course_url,
            "description": "Поглиблений курс з обраної тематики",
            "instructors": ["Microsoft Certified Trainer", "Industry Expert"],
            "total_duration": "8 годин",
            "modules_count": 6,
            "difficulty_level": "Intermediate",
            "last_updated": "2024-01-15",
            "completion_certificate": True,
            "languages": ["English", "Ukrainian subtitles"]
        }
        
        # Детальний syllabus
        if include_syllabus:
            course_data["syllabus"] = [
                {
                    "module": 1,
                    "title": "Вступ до тематики",
                    "duration": "45 хв",
                    "topics": ["Основні поняття", "Історія розвитку", "Сучасні тенденції"]
                },
                {
                    "module": 2,
                    "title": "Основи та принципи",
                    "duration": "1 година 20 хв",
                    "topics": ["Ключові концепції", "Принципи роботи", "Best Practices"]
                },
                {
                    "module": 3,
                    "title": "Практичне застосування",
                    "duration": "2 години",
                    "topics": ["Реальні кейси", "Практичні вправи", "Проектна робота"]
                },
                {
                    "module": 4,
                    "title": "Розширені техніки",
                    "duration": "1 година 45 хв",
                    "topics": ["Оптимізація", "Труднощі та вирішення", "Інтеграція"]
                },
                {
                    "module": 5,
                    "title": "Інструменти та технології",
                    "duration": "1 година 30 хв",
                    "topics": ["Popular Tools", "Framework Overview", "Technology Stack"]
                },
                {
                    "module": 6,
                    "title": "Фінальний проект та сертифікація",
                    "duration": "40 хв",
                    "topics": ["Project Submission", "Final Assessment", "Certification Process"]
                }
            ]
        
        # Вимоги та prerequisites
        if include_prerequisites:
            course_data["prerequisites"] = {
                "technical_skills": ["Базові знання програмування", "Робота з командним рядком"],
                "experience_level": "Початківець-Середній",
                "required_tools": ["Web Browser", "Text Editor"],
                "recommended_courses": ["Основи програмування", "Вступ до Computer Science"]
            }
        
        # Цілі навчання
        if include_learning_objectives:
            course_data["learning_objectives"] = [
                "Розуміння основних концепцій та принципів",
                "Здатність застосовувати знання на практиці",
                "Вміння вирішувати реальні проблеми",
                "Підготовка до сертифікаційних іспитів",
                "Розвиток критичного мислення в галузі"
            ]
        
        # Відгуки та рейтинги (симуляція)
        if include_reviews:
            course_data["reviews"] = {
                "average_rating": 4.7,
                "total_reviews": 1247,
                "review_summary": {
                    "5_stars": 68,
                    "4_stars": 25,
                    "3_stars": 5,
                    "2_stars": 1,
                    "1_star": 1
                },
                "sample_reviews": [
                    {
                        "user": "Анонімний студент",
                        "rating": 5,
                        "comment": "Чудовий курс з чіткою структурою та практичними прикладами.",
                        "date": "2024-01-10"
                    },
                    {
                        "user": "Розробник з 3-річним досвідом",
                        "rating": 4,
                        "comment": "Добрий матеріал для оновлення знань. Можна більше практики.",
                        "date": "2024-01-08"
                    }
                ]
            }
        
        # Додаткова мета-інформація
        course_data["additional_info"] = {
            "access_period": "Безстроковий",
            "downloadable_resources": True,
            "labs_and_exercises": True,
            "community_support": True,
            "mobile_access": True,
            "career_opportunities": ["Junior Developer", "Data Analyst", "IT Specialist"]
        }
        
        return json.dumps({
            "success": True,
            "course_details": course_data,
            "metadata": {
                "source": "Microsoft Learn Catalog",
                "retrieved_at": "2024-01-20",
                "data_completeness": "high"
            }
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка при отриманні деталей курсу: {str(e)}",
            "suggestion": "Перевірте URL курсу та спробуйте ще раз"
        }, ensure_ascii=False)

# Оновлюємо словник доступних функцій
available_functions = {
    "search_courses": search_courses,
    "get_user_preferences": get_user_preferences,
    "get_course_details": get_course_details,
}

# Приклад використання нової функції
example_messages = [
    {
        "role": "user",
        "content": "Знайди курси з AI для розробників, а потім покажи детальну інформацію про перший знайдений курс"
    }
]

В) Реалізуйте багатомовну підтримку Додайте можливість отримувати результати різними мовами:

In [22]:
# 🔧 Додаємо функцію для багатомовної підтримки
functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_courses",
            "description": "Returns a list of training courses from the Microsoft catalog with advanced filtering and multilingual support",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": "User role (for example: developer, student, administrator, data scientist)"
                    },
                    "subject": {
                        "type": "string",
                        "description": "Subject area (Artificial Intelligence, Data Science, Cloud Computing, Cybersecurity, Web Development, etc.)"
                    },
                    "level": {
                        "type": "string",
                        "description": "User experience level (beginner, intermediate, advanced)"
                    },
                    "language": {
                        "type": "string",
                        "description": "Content language (uk-ua, en-us, fr-fr, de-de, es-es, pl-pl, ru-ru, ja-jp, ko-kr, zh-cn)"
                    },
                    "output_language": {
                        "type": "string",
                        "description": "Language for displaying results and descriptions (ukrainian, english, french, german, spanish, polish, russian, japanese, korean, chinese)"
                    },
                    "translation_enabled": {
                        "type": "boolean",
                        "description": "Whether to provide translated descriptions and summaries"
                    }
                },
                "required": ["role", "subject"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "get_multilingual_course_details",
            "description": "Get detailed course information in multiple languages with translation support",
            "parameters": {
                "type": "object",
                "properties": {
                    "course_url": {
                        "type": "string",
                        "description": "URL of the course to get detailed information about"
                    },
                    "target_language": {
                        "type": "string",
                        "description": "Language for the output (ukrainian, english, french, german, spanish, etc.)"
                    },
                    "include_translations": {
                        "type": "boolean",
                        "description": "Whether to include translated content"
                    },
                    "multilingual_summary": {
                        "type": "boolean",
                        "description": "Whether to provide summary in multiple languages"
                    }
                },
                "required": ["course_url", "target_language"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "get_supported_languages",
            "description": "Get list of supported languages for courses and content",
            "parameters": {
                "type": "object",
                "properties": {
                    "subject": {
                        "type": "string",
                        "description": "Subject area to check language availability for"
                    },
                    "include_availability": {
                        "type": "boolean",
                        "description": "Whether to include course count per language"
                    }
                },
                "required": []
            }
        }
    )
]

def search_courses(role, subject, level=None, language="en-us", output_language="ukrainian", translation_enabled=True):
    """
    Функція пошуку курсів з багатомовною підтримкою
    """
    try:
        url = "https://learn.microsoft.com/api/catalog/"
        params = {
            "role": role,
            "subject": subject,
            "locale": language
        }
        
        if level:
            params["level"] = level
            
        print(f"🔍 Багатомовний пошук курсів для {role} з тематики {subject}...")
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        modules = data.get("modules", [])
        
        if not modules:
            error_messages = {
                "ukrainian": f"Не знайдено курсів для {role} з тематики {subject} мовою {language}",
                "english": f"No courses found for {role} in {subject} in {language}",
                "french": f"Aucun cours trouvé pour {role} en {subject} en {language}",
                "german": f"Keine Kurse gefunden für {role} in {subject} in {language}",
                "spanish": f"No se encontraron cursos para {role} en {subject} en {language}"
            }
            
            return json.dumps({
                "error": error_messages.get(output_language, error_messages["english"]),
                "suggestion": "Спробуйте іншу мову або тематику" if output_language == "ukrainian" else "Try different language or subject",
                "language_used": language,
                "output_language": output_language
            }, ensure_ascii=False)
        
        results = []
        for module in modules[:5]:
            title = module.get("title", "Без назви")
            url = module.get("url", "#")
            full_url = f"https://learn.microsoft.com{url}" if url.startswith('/') else url
            summary = module.get("summary", "Опис відсутній")
            
            # Багатомовні описи (симуляція перекладу)
            multilingual_descriptions = {}
            if translation_enabled:
                multilingual_descriptions = {
                    "ukrainian": f"Курс з {subject}: {summary}" if len(summary) > 50 else summary,
                    "english": f"Course on {subject}: {summary}",
                    "french": f"Cours sur {subject}: {summary}",
                    "german": f"Kurs über {subject}: {summary}",
                    "spanish": f"Curso sobre {subject}: {summary}"
                }
            
            course_data = {
                "title": title,
                "url": full_url,
                "summary": summary,
                "original_language": language,
                "duration": module.get("duration", "Невідомо"),
                "level": module.get("level", "Не вказано"),
                "multilingual_descriptions": multilingual_descriptions if translation_enabled else None,
                "available_languages": self.get_available_languages_for_course(module.get("uid")),
                "translated_summary": multilingual_descriptions.get(output_language, summary) if translation_enabled else summary
            }
            results.append(course_data)
        
        # Багатомовні відповіді
        success_messages = {
            "ukrainian": f"Знайдено {len(results)} курсів з {subject} для {role}",
            "english": f"Found {len(results)} courses on {subject} for {role}",
            "french": f"Trouvé {len(results)} cours sur {subject} pour {role}",
            "german": f"Gefunden {len(results)} Kurse über {subject} für {role}",
            "spanish": f"Encontrados {len(results)} cursos sobre {subject} para {role}"
        }
        
        return json.dumps({
            "success": True,
            "message": success_messages.get(output_language, success_messages["english"]),
            "search_parameters": {
                "role": role,
                "subject": subject,
                "level": level,
                "content_language": language,
                "output_language": output_language
            },
            "courses_found": len(results),
            "courses": results,
            "language_info": {
                "content_language": language,
                "output_language": output_language,
                "translation_enabled": translation_enabled
            }
        }, ensure_ascii=False)
        
    except Exception as e:
        error_messages = {
            "ukrainian": f"Помилка пошуку: {str(e)}",
            "english": f"Search error: {str(e)}",
            "french": f"Erreur de recherche: {str(e)}",
            "german": f"Suchfehler: {str(e)}",
            "spanish": f"Error de búsqueda: {str(e)}"
        }
        
        return json.dumps({
            "error": error_messages.get(output_language, error_messages["english"]),
            "courses_found": 0
        }, ensure_ascii=False)

def get_multilingual_course_details(course_url, target_language="ukrainian", include_translations=True, multilingual_summary=True):
    """
    Отримання детальної інформації про курс з багатомовною підтримкою
    """
    try:
        print(f"🔍 Отримання багатомовних деталей курсу: {course_url}")
        
        # Симуляція багатомовного контенту
        course_details = {
            "title": {
                "original": "Advanced Artificial Intelligence Course",
                "ukrainian": "Просунутий курс з Штучного Інтелекту",
                "english": "Advanced Artificial Intelligence Course",
                "french": "Cours avancé d'intelligence artificielle",
                "german": "Fortgeschrittener KI-Kurs"
            },
            "description": {
                "original": "Comprehensive course covering advanced AI concepts and applications",
                "ukrainian": "Комплексний курс, що охоплює передові концепції та застосування ШІ",
                "english": "Comprehensive course covering advanced AI concepts and applications",
                "french": "Cours complet couvrant les concepts et applications avancés de l'IA",
                "german": "Umfassender Kurs zu fortgeschrittenen KI-Konzepten und Anwendungen"
            },
            "instructors": ["Microsoft Certified Trainer", "AI Specialist"],
            "duration": "8 hours",
            "level": "Intermediate",
            "languages_available": ["en-us", "uk-ua", "fr-fr", "de-de", "es-es"],
            "modules": [
                {
                    "title": {
                        "original": "Introduction to AI",
                        "ukrainian": "Вступ до ШІ",
                        "english": "Introduction to AI",
                        "french": "Introduction à l'IA"
                    },
                    "duration": "45 min"
                }
            ]
        }
        
        # Формуємо відповідь у вибраній мові
        response_data = {
            "course_url": course_url,
            "target_language": target_language,
            "title": course_details["title"].get(target_language, course_details["title"]["original"]),
            "description": course_details["description"].get(target_language, course_details["description"]["original"]),
            "instructors": course_details["instructors"],
            "duration": course_details["duration"],
            "level": course_details["level"],
            "available_languages": course_details["languages_available"]
        }
        
        if include_translations:
            response_data["multilingual_content"] = course_details
        
        if multilingual_summary:
            response_data["summary"] = self.generate_multilingual_summary(course_details, target_language)
        
        return json.dumps({
            "success": True,
            "course_details": response_data,
            "language_support": {
                "target_language": target_language,
                "translations_included": include_translations,
                "multilingual_summary": multilingual_summary
            }
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Error getting course details: {str(e)}",
            "suggestion": "Check course URL and try again"
        }, ensure_ascii=False)

def get_supported_languages(subject=None, include_availability=True):
    """
    Отримання списку підтримуваних мов
    """
    try:
        supported_languages = {
            "uk-ua": {"name": "Ukrainian", "courses_count": 1500},
            "en-us": {"name": "English", "courses_count": 5000},
            "fr-fr": {"name": "French", "courses_count": 1200},
            "de-de": {"name": "German", "courses_count": 1100},
            "es-es": {"name": "Spanish", "courses_count": 1300},
            "pl-pl": {"name": "Polish", "courses_count": 800},
            "ru-ru": {"name": "Russian", "courses_count": 1400},
            "ja-jp": {"name": "Japanese", "courses_count": 900},
            "ko-kr": {"name": "Korean", "courses_count": 700},
            "zh-cn": {"name": "Chinese", "courses_count": 1000}
        }
        
        # Фільтрація за тематикою (симуляція)
        if subject:
            for lang in supported_languages:
                supported_languages[lang]["courses_count"] = max(100, supported_languages[lang]["courses_count"] // 5)
        
        response_data = {
            "supported_languages": supported_languages,
            "total_languages": len(supported_languages)
        }
        
        if not include_availability:
            for lang in supported_languages:
                if "courses_count" in supported_languages[lang]:
                    del supported_languages[lang]["courses_count"]
        
        return json.dumps({
            "success": True,
            "language_support": response_data,
            "subject_filter": subject
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Error getting supported languages: {str(e)}"
        }, ensure_ascii=False)

# Допоміжні функції для багатомовної підтримки
def get_available_languages_for_course(course_id):
    """Симуляція отримання доступних мов для курсу"""
    return ["en-us", "uk-ua", "fr-fr"]

def generate_multilingual_summary(course_details, target_language):
    """Генерація багатомовного резюме"""
    summaries = {
        "ukrainian": f"Курс '{course_details['title']['ukrainian']}' тривалістю {course_details['duration']}",
        "english": f"Course '{course_details['title']['english']}' with duration {course_details['duration']}",
        "french": f"Cours '{course_details['title']['french']}' d'une durée de {course_details['duration']}"
    }
    return summaries.get(target_language, summaries["english"])

# Оновлюємо словник доступних функцій
available_functions = {
    "search_courses": search_courses,
    "get_multilingual_course_details": get_multilingual_course_details,
    "get_supported_languages": get_supported_languages,
}

# Приклади використання багатомовного функціоналу
example_messages = [
    {
        "role": "user", 
        "content": "Знайди курси з Data Science українською мовою для початківців"
    },
    {
        "role": "user",
        "content": "Find AI courses in German for intermediate level developers"
    },
    {
        "role": "user",
        "content": "Покажи які мови підтримуються для курсів з Cybersecurity"
    }
]

Г) Аналіз даних та рекомендації Створіть функцію аналізу курсів для надання рекомендацій:

In [23]:
# 🔧 Додаємо функції для аналізу даних та рекомендацій
functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_courses",
            "description": "Returns a list of training courses from the Microsoft catalog with advanced filtering and analytics",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": "User role (for example: developer, student, administrator, data scientist)"
                    },
                    "subject": {
                        "type": "string",
                        "description": "Subject area (Artificial Intelligence, Data Science, Cloud Computing, Cybersecurity, Web Development, etc.)"
                    },
                    "level": {
                        "type": "string",
                        "description": "User experience level (beginner, intermediate, advanced)"
                    }
                },
                "required": ["role", "subject"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "analyze_courses_and_recommend",
            "description": "Analyze available courses and provide personalized recommendations based on user profile and goals",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_role": {
                        "type": "string",
                        "description": "Current role or career position"
                    },
                    "target_role": {
                        "type": "string",
                        "description": "Desired future role or career goal"
                    },
                    "current_skills": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of current skills and technologies"
                    },
                    "target_skills": {
                        "type": "array", 
                        "items": {"type": "string"},
                        "description": "List of skills user wants to acquire"
                    },
                    "learning_style": {
                        "type": "string",
                        "description": "Preferred learning style (visual, hands-on, theoretical, mixed)"
                    },
                    "time_availability": {
                        "type": "string",
                        "description": "Available learning time per week (low: <5h, medium: 5-10h, high: >10h)"
                    },
                    "priority_areas": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "High priority learning areas"
                    },
                    "budget_constraint": {
                        "type": "string",
                        "description": "Budget for learning (free, low, medium, high)"
                    }
                },
                "required": ["user_role", "target_role", "current_skills", "target_skills"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "get_learning_path_analysis",
            "description": "Analyze and compare different learning paths for skill development",
            "parameters": {
                "type": "object",
                "properties": {
                    "career_goal": {
                        "type": "string",
                        "description": "Target career position or role"
                    },
                    "timeframe": {
                        "type": "string", 
                        "description": "Desired timeframe for achievement (1month, 3months, 6months, 1year)"
                    },
                    "current_level": {
                        "type": "string",
                        "description": "Current skill level in target area"
                    },
                    "learning_intensity": {
                        "type": "string",
                        "description": "Learning intensity preference (casual, regular, intensive)"
                    },
                    "include_certifications": {
                        "type": "boolean",
                        "description": "Whether to include certification preparation"
                    }
                },
                "required": ["career_goal", "timeframe"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "get_market_trend_analysis", 
            "description": "Get analysis of market trends and in-demand skills for career planning",
            "parameters": {
                "type": "object",
                "properties": {
                    "industry": {
                        "type": "string",
                        "description": "Target industry or sector"
                    },
                    "region": {
                        "type": "string",
                        "description": "Geographic region for market analysis"
                    },
                    "time_horizon": {
                        "type": "string",
                        "description": "Time horizon for trend analysis (short_term, medium_term, long_term)"
                    },
                    "include_salary_data": {
                        "type": "boolean", 
                        "description": "Whether to include salary information"
                    },
                    "skill_categories": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Specific skill categories to analyze"
                    }
                },
                "required": ["industry"]
            }
        }
    )
]

def analyze_courses_and_recommend(user_role, target_role, current_skills, target_skills, learning_style="mixed", 
                                 time_availability="medium", priority_areas=None, budget_constraint="free"):
    """
    Аналіз курсів та надання персоналізованих рекомендацій
    """
    try:
        # Симуляція аналізу навичок
        skill_gap_analysis = analyze_skill_gap(current_skills, target_skills)
        
        # Пошук релевантних курсів
        recommended_courses = find_relevant_courses(target_skills, user_role)
        
        # Аналіз оптимального шляху навчання
        learning_path = create_optimal_learning_path(recommended_courses, time_availability, learning_style)
        
        # Розрахунок часу та ресурсів
        time_estimation = calculate_time_requirements(learning_path, time_availability)
        
        # Оцінка кар'єрного потенціалу
        career_impact = assess_career_impact(user_role, target_role, skill_gap_analysis)
        
        # Генерація персоналізованих рекомендацій
        recommendations = generate_personalized_recommendations(
            learning_path, skill_gap_analysis, career_impact, budget_constraint
        )
        
        return json.dumps({
            "success": True,
            "user_profile": {
                "current_role": user_role,
                "target_role": target_role,
                "learning_style": learning_style,
                "time_availability": time_availability,
                "budget_constraint": budget_constraint
            },
            "skill_analysis": skill_gap_analysis,
            "learning_path": learning_path,
            "time_estimation": time_estimation,
            "career_impact": career_impact,
            "recommendations": recommendations,
            "metrics": {
                "skill_gap_score": skill_gap_analysis["gap_score"],
                "estimated_completion_time": time_estimation["total_weeks"],
                "career_improvement_potential": career_impact["improvement_score"],
                "confidence_level": "high"
            }
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка аналізу: {str(e)}",
            "suggestion": "Спробуйте оновити параметри аналізу"
        }, ensure_ascii=False)

def get_learning_path_analysis(career_goal, timeframe, current_level="beginner", learning_intensity="regular", include_certifications=True):
    """
    Аналіз та порівняння навчальних шляхів
    """
    try:
        # Визначення оптимального шляху на основі цілей
        learning_paths = generate_learning_paths(career_goal, current_level)
        
        # Фільтрація за часовими рамками
        filtered_paths = filter_paths_by_timeframe(learning_paths, timeframe, learning_intensity)
        
        # Аналіз варіантів
        path_comparison = compare_learning_paths(filtered_paths, include_certifications)
        
        # Рекомендації
        optimal_path = select_optimal_path(path_comparison, learning_intensity)
        
        return json.dumps({
            "success": True,
            "analysis_parameters": {
                "career_goal": career_goal,
                "timeframe": timeframe,
                "current_level": current_level,
                "learning_intensity": learning_intensity
            },
            "available_paths": learning_paths,
            "filtered_paths": filtered_paths,
            "path_comparison": path_comparison,
            "recommended_path": optimal_path,
            "key_metrics": {
                "total_paths_analyzed": len(learning_paths),
                "paths_matching_timeframe": len(filtered_paths),
                "estimated_success_probability": optimal_path["success_probability"],
                "resource_requirements": optimal_path["resource_requirements"]
            }
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка аналізу навчальних шляхів: {str(e)}"
        }, ensure_ascii=False)

def get_market_trend_analysis(industry, region="global", time_horizon="medium_term", include_salary_data=True, skill_categories=None):
    """
    Аналіз ринкових трендів та затребуваних навичок
    """
    try:
        # Отримання даних про тренди
        market_trends = get_industry_trends(industry, region, time_horizon)
        
        # Аналіз затребуваних навичок
        in_demand_skills = analyze_demand_skills(market_trends, skill_categories)
        
        # Дані про зарплати (якщо потрібно)
        salary_insights = {}
        if include_salary_data:
            salary_insights = get_salary_insights(industry, region, in_demand_skills)
        
        # Рекомендації для навчання
        learning_recommendations = generate_market_aligned_recommendations(in_demand_skills, market_trends)
        
        return json.dumps({
            "success": True,
            "market_analysis": {
                "industry": industry,
                "region": region,
                "time_horizon": time_horizon,
                "growth_outlook": market_trends["growth_outlook"]
            },
            "in_demand_skills": in_demand_skills,
            "salary_insights": salary_insights,
            "learning_recommendations": learning_recommendations,
            "risk_assessment": {
                "market_volatility": market_trends["volatility"],
                "skill_demand_stability": in_demand_skills["stability_score"],
                "future_proof_score": market_trends["future_proof_score"]
            }
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка аналізу ринкових трендів: {str(e)}"
        }, ensure_ascii=False)

# Допоміжні функції для аналізу даних
def analyze_skill_gap(current_skills, target_skills):
    """Аналіз розриву між поточними та цільовими навичками"""
    missing_skills = [skill for skill in target_skills if skill not in current_skills]
    existing_skills = [skill for skill in target_skills if skill in current_skills]
    
    return {
        "missing_skills": missing_skills,
        "existing_skills": existing_skills,
        "gap_score": len(missing_skills) / len(target_skills) if target_skills else 0,
        "coverage_percentage": len(existing_skills) / len(target_skills) * 100 if target_skills else 0
    }

def find_relevant_courses(target_skills, user_role):
    """Пошук курсів, релевантних цільовим навичкам"""
    # Симуляція даних курсів
    courses_db = {
        "Python": [
            {"title": "Python for Beginners", "level": "beginner", "duration": "10h", "relevance": 0.9},
            {"title": "Advanced Python Programming", "level": "advanced", "duration": "15h", "relevance": 0.8}
        ],
        "Machine Learning": [
            {"title": "ML Fundamentals", "level": "intermediate", "duration": "12h", "relevance": 0.95},
            {"title": "Deep Learning Specialization", "level": "advanced", "duration": "20h", "relevance": 0.85}
        ],
        "Data Analysis": [
            {"title": "Data Analysis with Python", "level": "intermediate", "duration": "8h", "relevance": 0.88}
        ]
    }
    
    relevant_courses = []
    for skill in target_skills:
        if skill in courses_db:
            relevant_courses.extend(courses_db[skill])
    
    return sorted(relevant_courses, key=lambda x: x["relevance"], reverse=True)[:10]

def create_optimal_learning_path(courses, time_availability, learning_style):
    """Створення оптимального шляху навчання"""
    intensity_multiplier = {"low": 1.5, "medium": 1.0, "high": 0.7}
    base_weeks = sum([parse_duration(course["duration"]) for course in courses]) / 10  # 10h на тиждень
    
    return {
        "courses_sequence": courses,
        "estimated_total_duration": f"{base_weeks:.1f} тижнів",
        "weekly_commitment": f"{10 * intensity_multiplier.get(time_availability, 1.0):.1f} годин",
        "learning_strategy": adapt_learning_strategy(learning_style),
        "milestones": generate_learning_milestones(courses)
    }

def assess_career_impact(current_role, target_role, skill_gap):
    """Оцінка впливу на кар'єру"""
    role_transitions = {
        "student": {"developer": 0.8, "data_scientist": 0.6, "administrator": 0.4},
        "developer": {"senior_developer": 0.9, "data_scientist": 0.7, "team_lead": 0.6},
        "data_analyst": {"data_scientist": 0.8, "ml_engineer": 0.7, "analytics_manager": 0.6}
    }
    
    transition_probability = role_transitions.get(current_role, {}).get(target_role, 0.5)
    improvement_score = transition_probability * (1 - skill_gap["gap_score"])
    
    return {
        "transition_probability": transition_probability,
        "improvement_score": improvement_score,
        "salary_increase_potential": improvement_score * 0.3,  # Симуляція
        "career_acceleration": improvement_score * 0.4
    }

def generate_personalized_recommendations(learning_path, skill_gap, career_impact, budget):
    """Генерація персоналізованих рекомендацій"""
    recommendations = []
    
    if skill_gap["gap_score"] > 0.7:
        recommendations.append("Рекомендується почати з фундаментальних курсів для заповнення основних розривів у навичках")
    
    if career_impact["improvement_score"] > 0.8:
        recommendations.append("Високий потенціал кар'єрного зростання - рекомендується інтенсивне навчання")
    
    if budget == "free":
        recommendations.append("Доступні безкоштовні альтернативи для всіх рекомендованих курсів")
    
    return recommendations

# Оновлюємо словник доступних функцій
available_functions = {
    "search_courses": search_courses,
    "analyze_courses_and_recommend": analyze_courses_and_recommend,
    "get_learning_path_analysis": get_learning_path_analysis,
    "get_market_trend_analysis": get_market_trend_analysis,
}

# Приклади використання аналітичних функцій
example_messages = [
    {
        "role": "user",
        "content": "Проаналізуй мої навички та рекомендуй курси для переходу з розробника на data scientist"
    },
    {
        "role": "user", 
        "content": "Покажи аналіз ринкових трендів для AI індустрії та рекомендовані навички"
    },
    {
        "role": "user",
        "content": "Створи оптимальний шлях навчання для ML engineer з часовим обмеженням 3 місяці"
    }
]

In [26]:
# 🧠 Тестувальний запит для перевірки аналітичних функцій
messages = [
    {
        "role": "user",
        "content": "Я розробник з навичками Python та SQL. Хочу стати Data Scientist. Проаналізуй мої навички та рекомендуй курси для цього переходу."
    }
]

response = client.complete(
    model=deployment,
    messages=messages,
    tools=functions,
    tool_choice="auto"
)

response_message = response.choices[0].message
print("📥 Відповідь моделі:")
print(response_message)

# Додаткова інформація для дебагінгу
print("\n🔍 Детальна інформація про відповідь:")
print(f"Завершення причини: {response.choices[0].finish_reason}")
print(f"Використані токени: {response.usage.total_tokens if hasattr(response, 'usage') else 'N/A'}")

# Перевірка, чи модель хоче викликати функції
if response_message.tool_calls:
    print(f"\n🛠️ Модель хоче викликати {len(response_message.tool_calls)} функцій:")
    for i, tool_call in enumerate(response_message.tool_calls):
        print(f"Функція {i+1}: {tool_call.function.name}")
        print(f"Аргументи: {tool_call.function.arguments}")
else:
    print("\nℹ️ Модель відповіла без виклику функцій")

📥 Відповідь моделі:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"user_role":"розробник","target_role":"Data Scientist","current_skills":["Python","SQL"],"target_skills":["машинне навчання","обробка даних","статистика","алгоритми штучного інтелекту","аналіз даних"],"learning_style":"змішаний","time_availability":"середній","priority_areas":["машинне навчання","аналіз даних","алгоритми штучного інтелекту"],"budget_constraint":"low"}', 'name': 'analyze_courses_and_recommend'}, 'id': 'call_uck25st8HOnIJ2GuWGsZKeRO', 'type': 'function'}]}

🔍 Детальна інформація про відповідь:
Завершення причини: CompletionsFinishReason.TOOL_CALLS
Використані токени: 616

🛠️ Модель хоче викликати 1 функцій:
Функція 1: analyze_courses_and_recommend
Аргументи: {"user_role":"розробник","target_role":"Data Scientist","current_skills":["Python","SQL"],"target_skills":["машинне навчання","обробка даних","статистика","алгоритми штучного інте

In [25]:
# Запуск тестів
test_results = []
# Функція для тестування різних запитів
def test_model_response(test_messages, test_name):
    print(f"\n{'='*50}")
    print(f"🧪 ТЕСТ: {test_name}")
    print(f"{'='*50}")
    
    response = client.complete(
        model=deployment,
        messages=test_messages,
        tools=functions,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    print("📥 Відповідь моделі:")
    print(response_message)
    
    if response_message.tool_calls:
        print(f"\n✅ Модель просить викликати функції:")
        for tool_call in response_message.tool_calls:
            print(f"   - {tool_call.function.name}")
    else:
        print(f"\nℹ️ Модель надала пряму відповідь")
    
    return response_message
# Тест пошуку курсів
messages = [
    {
        "role": "user", 
        "content": "Знайди курси з Machine Learning для початківців"
    }
]
test_results.append(test_model_response(messages, "Пошук курсів ML"))

# Тест аналізу ринкових трендів
message = [
    {
        "role": "user",
        "content": "Які навички зараз найбільш затребувані в AI індустрії?"
    }
]
test_results.append(test_model_response(messages, "Аналіз ринкових трендів"))
# Тест аналізу навчального шляху
messages = [
    {
        "role": "user",
        "content": "Створи план навчання для ML engineer на 6 місяців"
    }
]
test_results.append(test_model_response(messages, "План навчання ML engineer"))


🧪 ТЕСТ: Пошук курсів ML
📥 Відповідь моделі:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"role":"student","subject":"Machine Learning","level":"beginner"}', 'name': 'search_courses'}, 'id': 'call_jPgy96sQ7hi4P0RXdBUfWDP9', 'type': 'function'}]}

✅ Модель просить викликати функції:
   - search_courses

🧪 ТЕСТ: Аналіз ринкових трендів
📥 Відповідь моделі:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"role":"student","subject":"Machine Learning","level":"beginner"}', 'name': 'search_courses'}, 'id': 'call_y86EnfVwrb3ousXnopW035RY', 'type': 'function'}]}

✅ Модель просить викликати функції:
   - search_courses

🧪 ТЕСТ: План навчання ML engineer
📥 Відповідь моделі:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"career_goal":"ML engineer","timeframe":"6months","learning_inte

___

## Індивиідуальне завдання, варінат №3. Тема: "devops"

In [27]:
import os
import json
import requests
from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import ChatCompletionsToolDefinition
from azure.core.credentials import AzureKeyCredential

load_dotenv()
token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.inference.ai.azure.com"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

# Виберіть модель загального призначення для тексту
deployment = "gpt-4o"

In [28]:
# 🔧 Визначення функцій для DevOps тематики
functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_devops_courses",
            "description": "Search for DevOps-related courses and learning paths from Microsoft Learn catalog",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": "Current role (developer, sysadmin, operations, student, etc.)"
                    },
                    "experience_level": {
                        "type": "string",
                        "description": "Experience level in DevOps (beginner, intermediate, advanced)"
                    },
                    "technology_focus": {
                        "type": "string",
                        "description": "Specific technology or area (Azure, Docker, Kubernetes, CI/CD, Terraform, Ansible, Monitoring)"
                    },
                    "certification_goal": {
                        "type": "string",
                        "description": "Target certification (AZ-400, AZ-104, AZ-204, AZ-305, etc.)"
                    },
                    "content_type": {
                        "type": "string",
                        "description": "Type of content (course, learning_path, lab, certification)"
                    }
                },
                "required": ["role"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "analyze_devops_skills_gap",
            "description": "Analyze DevOps skills gap and provide personalized learning recommendations",
            "parameters": {
                "type": "object",
                "properties": {
                    "current_role": {
                        "type": "string",
                        "description": "Current job role or position"
                    },
                    "target_devops_role": {
                        "type": "string",
                        "description": "Target DevOps role (DevOps Engineer, SRE, Platform Engineer, Cloud Engineer)"
                    },
                    "current_skills": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of current technical skills"
                    },
                    "infrastructure_experience": {
                        "type": "string",
                        "description": "Experience with infrastructure (none, basic, intermediate, advanced)"
                    },
                    "cloud_platforms": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Familiar cloud platforms (Azure, AWS, GCP, on-premise)"
                    },
                    "coding_experience": {
                        "type": "string",
                        "description": "Programming experience level"
                    }
                },
                "required": ["current_role", "target_devops_role", "current_skills"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "get_devops_learning_roadmap",
            "description": "Generate a comprehensive DevOps learning roadmap based on career goals",
            "parameters": {
                "type": "object",
                "properties": {
                    "career_stage": {
                        "type": "string",
                        "description": "Current career stage (starting, transitioning, advancing)"
                    },
                    "time_commitment": {
                        "type": "string",
                        "description": "Available learning time (part_time: 5-10h/week, full_time: 20+h/week, intensive: 40+h/week)"
                    },
                    "focus_areas": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Key focus areas (CI/CD, Infrastructure as Code, Monitoring, Security, Cloud Platforms)"
                    },
                    "target_timeframe": {
                        "type": "string",
                        "description": "Target completion timeframe (1month, 3months, 6months, 1year)"
                    },
                    "include_certifications": {
                        "type": "boolean",
                        "description": "Include certification preparation in roadmap"
                    }
                },
                "required": ["career_stage", "time_commitment"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "get_devops_tools_recommendations",
            "description": "Get recommendations for DevOps tools and technologies based on specific needs",
            "parameters": {
                "type": "object",
                "properties": {
                    "use_case": {
                        "type": "string",
                        "description": "Primary use case (web_apps, microservices, data_pipelines, mobile_apps, legacy_systems)"
                    },
                    "team_size": {
                        "type": "string",
                        "description": "Team size organization (startup: 1-10, small: 11-50, medium: 51-200, large: 201+)"
                    },
                    "budget_constraint": {
                        "type": "string",
                        "description": "Budget considerations (free_open_source, low_cost, enterprise)"
                    },
                    "cloud_preference": {
                        "type": "string",
                        "description": "Cloud platform preference (Azure, AWS, GCP, multi_cloud, hybrid)"
                    },
                    "automation_focus": {
                        "type": "string",
                        "description": "Automation priority (basic, moderate, extensive)"
                    }
                },
                "required": ["use_case", "team_size"]
            }
        }
    )
]

In [29]:
# Функції для DevOps тематики
def search_devops_courses(role, experience_level=None, technology_focus=None, certification_goal=None, content_type=None):
    """
    Пошук DevOps курсів та навчальних матеріалів
    """
    try:
        # Базові параметри для DevOps пошуку
        params = {
            "role": role,
            "subject": "DevOps"
        }
        
        if experience_level:
            params["level"] = experience_level
        if technology_focus:
            params["technology"] = technology_focus
            
        print(f"🔍 Пошук DevOps курсів для {role}...")
        
        # Симуляція результатів пошуку
        devops_courses = {
            "beginner": [
                {
                    "title": "Introduction to DevOps",
                    "url": "https://learn.microsoft.com/devops/learn",
                    "duration": "4h",
                    "level": "Beginner",
                    "type": "learning_path",
                    "technologies": ["Azure DevOps", "Git", "CI/CD"],
                    "description": "Основи DevOps практик та принципів"
                },
                {
                    "title": "Azure Fundamentals",
                    "url": "https://learn.microsoft.com/azure/fundamentals",
                    "duration": "6h", 
                    "level": "Beginner",
                    "type": "course",
                    "technologies": ["Azure", "Cloud Computing"],
                    "description": "Основи хмарних обчислень з Azure"
                }
            ],
            "intermediate": [
                {
                    "title": "Azure DevOps CI/CD Pipelines",
                    "url": "https://learn.microsoft.com/azure/devops/pipelines",
                    "duration": "8h",
                    "level": "Intermediate", 
                    "type": "learning_path",
                    "technologies": ["Azure DevOps", "YAML", "Pipelines"],
                    "description": "Створення та управління CI/CD пайплайнами"
                },
                {
                    "title": "Docker and Container Fundamentals",
                    "url": "https://learn.microsoft.com/containers",
                    "duration": "5h",
                    "level": "Intermediate",
                    "type": "course", 
                    "technologies": ["Docker", "Containers", "Kubernetes"],
                    "description": "Робота з контейнерами та Docker"
                }
            ],
            "advanced": [
                {
                    "title": "Kubernetes Advanced Patterns",
                    "url": "https://learn.microsoft.com/azure/aks",
                    "duration": "10h",
                    "level": "Advanced",
                    "type": "learning_path",
                    "technologies": ["Kubernetes", "AKS", "Helm"],
                    "description": "Розширені патерни Kubernetes для продакшену"
                },
                {
                    "title": "Infrastructure as Code with Terraform",
                    "url": "https://learn.microsoft.com/devops/develop/terraform",
                    "duration": "7h",
                    "level": "Advanced",
                    "type": "course",
                    "technologies": ["Terraform", "IaC", "Azure"],
                    "description": "Автоматизація інфраструктури з Terraform"
                }
            ]
        }
        
        # Фільтрація за рівнем
        filtered_courses = []
        if experience_level and experience_level in devops_courses:
            filtered_courses = devops_courses[experience_level]
        else:
            # Якщо рівень не вказано, показуємо всі
            for level_courses in devops_courses.values():
                filtered_courses.extend(level_courses)
        
        # Додаткова фільтрація за технологією
        if technology_focus:
            filtered_courses = [course for course in filtered_courses 
                              if technology_focus.lower() in [tech.lower() for tech in course.get("technologies", [])]]
        
        # Фільтрація за типом контенту
        if content_type:
            filtered_courses = [course for course in filtered_courses 
                              if course.get("type", "").lower() == content_type.lower()]
        
        if not filtered_courses:
            return json.dumps({
                "error": f"Не знайдено DevOps курсів для вказаних критеріїв",
                "suggestion": "Спробуйте змінити рівень досвіду або технологічний фокус"
            }, ensure_ascii=False)
        
        return json.dumps({
            "success": True,
            "search_criteria": {
                "role": role,
                "experience_level": experience_level,
                "technology_focus": technology_focus,
                "certification_goal": certification_goal,
                "content_type": content_type
            },
            "courses_found": len(filtered_courses),
            "courses": filtered_courses[:8],  # Обмежуємо кількість результатів
            "recommendations": generate_devops_recommendations(role, experience_level)
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка пошуку DevOps курсів: {str(e)}",
            "courses_found": 0
        }, ensure_ascii=False)

def analyze_devops_skills_gap(current_role, target_devops_role, current_skills, infrastructure_experience="none", cloud_platforms=None, coding_experience="basic"):
    """
    Аналіз розриву навичок для переходу в DevOps
    """
    try:
        # Визначення необхідних навичок для різних DevOps ролей
        required_skills_map = {
            "DevOps Engineer": ["CI/CD", "Infrastructure as Code", "Containerization", "Monitoring", "Cloud Platforms", "Scripting"],
            "SRE": ["Monitoring", "Reliability Engineering", "Incident Management", "Capacity Planning", "Performance Optimization"],
            "Platform Engineer": ["Kubernetes", "Infrastructure Automation", "Developer Tools", "Platform Design", "Security"],
            "Cloud Engineer": ["Cloud Services", "Networking", "Security", "Cost Optimization", "Migration Strategies"]
        }
        
        required_skills = required_skills_map.get(target_devops_role, [])
        missing_skills = [skill for skill in required_skills if skill not in current_skills]
        
        # Оцінка розриву
        gap_score = len(missing_skills) / len(required_skills) if required_skills else 1.0
        
        # Генерація рекомендацій
        recommendations = generate_skills_recommendations(
            current_role, target_devops_role, missing_skills, 
            infrastructure_experience, cloud_platforms, coding_experience
        )
        
        return json.dumps({
            "success": True,
            "analysis": {
                "current_role": current_role,
                "target_role": target_devops_role,
                "gap_score": f"{gap_score:.2f}",
                "skills_coverage": f"{(1 - gap_score) * 100:.1f}%"
            },
            "skills_assessment": {
                "current_skills": current_skills,
                "required_skills": required_skills,
                "missing_skills": missing_skills,
                "strong_areas": [skill for skill in required_skills if skill in current_skills]
            },
            "recommendations": recommendations,
            "learning_priority": prioritize_learning_path(missing_skills, infrastructure_experience)
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка аналізу навичок: {str(e)}"
        }, ensure_ascii=False)

def get_devops_learning_roadmap(career_stage, time_commitment, focus_areas=None, target_timeframe="6months", include_certifications=True):
    """
    Генерація навчального плану для DevOps
    """
    try:
        # Визначення roadmap на основі кар'єрного етапу
        roadmaps = {
            "starting": generate_beginner_roadmap(time_commitment, focus_areas),
            "transitioning": generate_transition_roadmap(time_commitment, focus_areas),
            "advancing": generate_advanced_roadmap(time_commitment, focus_areas)
        }
        
        roadmap = roadmaps.get(career_stage, roadmaps["starting"])
        
        # Додавання сертифікацій
        if include_certifications:
            roadmap["certifications"] = get_recommended_certifications(career_stage, focus_areas)
        
        # Адаптація за часовими рамками
        adapted_roadmap = adapt_roadmap_to_timeframe(roadmap, target_timeframe, time_commitment)
        
        return json.dumps({
            "success": True,
            "roadmap_parameters": {
                "career_stage": career_stage,
                "time_commitment": time_commitment,
                "target_timeframe": target_timeframe,
                "focus_areas": focus_areas or ["CI/CD", "Cloud Platforms", "Automation"]
            },
            "learning_roadmap": adapted_roadmap,
            "estimated_timeline": calculate_timeline(adapted_roadmap, time_commitment),
            "key_milestones": extract_milestones(adapted_roadmap)
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка генерації roadmap: {str(e)}"
        }, ensure_ascii=False)

def get_devops_tools_recommendations(use_case, team_size, budget_constraint="free_open_source", cloud_preference=None, automation_focus="moderate"):
    """
    Рекомендації DevOps інструментів та технологій
    """
    try:
        tools_recommendations = generate_tools_recommendations(
            use_case, team_size, budget_constraint, cloud_preference, automation_focus
        )
        
        return json.dumps({
            "success": True,
            "recommendation_context": {
                "use_case": use_case,
                "team_size": team_size,
                "budget": budget_constraint,
                "cloud_preference": cloud_preference,
                "automation_level": automation_focus
            },
            "tools_stack": tools_recommendations,
            "implementation_priority": prioritize_tools_implementation(tools_recommendations),
            "learning_resources": get_tools_learning_resources(tools_recommendations)
        }, ensure_ascii=False)
        
    except Exception as e:
        return json.dumps({
            "error": f"Помилка генерації рекомендацій інструментів: {str(e)}"
        }, ensure_ascii=False)

# Допоміжні функції для DevOps
def generate_devops_recommendations(role, experience_level):
    """Генерація рекомендацій для DevOps навчання"""
    recommendations = {
        "developer": "Зосередьтеся на CI/CD, інфраструктурі як код та контейнеризації",
        "sysadmin": "Розвивайте навички автоматизації, хмарних платформ та моніторингу",
        "student": "Почніть з основ DevOps, Git та хмарних обчислень",
        "operations": "Вивчайте автоматизацію, моніторинг та управління інцидентами"
    }
    
    return recommendations.get(role, "Рекомендується комплексний підхід до DevOps практик")

def generate_skills_recommendations(current_role, target_role, missing_skills, infra_exp, cloud_platforms, coding_exp):
    """Генерація рекомендацій щодо навичок"""
    recommendations = []
    
    if "CI/CD" in missing_skills:
        recommendations.append("Рекомендується вивчити Azure DevOps, GitHub Actions або Jenkins")
    
    if "Containerization" in missing_skills:
        recommendations.append("Почніть з Docker, потім перейдіть до Kubernetes")
    
    if not cloud_platforms:
        recommendations.append("Оберіть хмарну платформу (Azure рекомендується для DevOps)")
    
    if coding_exp == "basic" and target_role in ["DevOps Engineer", "Platform Engineer"]:
        recommendations.append("Покращте навички програмування (Python, Bash, PowerShell)")
    
    return recommendations

def generate_beginner_roadmap(time_commitment, focus_areas):
    """Roadmap для початківців"""
    return {
        "phase_1": {
            "duration": "1-2 months",
            "topics": ["DevOps Principles", "Version Control with Git", "Basic Linux/Windows Administration"],
            "tools": ["Git", "GitHub", "Basic CLI"]
        },
        "phase_2": {
            "duration": "2-3 months", 
            "topics": ["CI/CD Fundamentals", "Container Basics", "Cloud Introduction"],
            "tools": ["Docker", "Azure DevOps", "Azure Fundamentals"]
        }
    }

In [36]:
# Словник доступних функцій
available_functions = {
    "search_devops_courses": search_devops_courses,
    "analyze_devops_skills_gap": analyze_devops_skills_gap,
    "get_devops_learning_roadmap": get_devops_learning_roadmap,
    "get_devops_tools_recommendations": get_devops_tools_recommendations,
}

# 🧠 Тестувальні запити для DevOps тематики
messages = [
    {
        "role": "user",
        "content": "Я розробник, хочу перейти в DevOps. Знайди курси для початківців з CI/CD та Docker"
    }
]

# Виклик моделі
response = client.complete(
    model=deployment,
    messages=messages,
    tools=functions,
    tool_choice="auto"
)

response_message = response.choices[0].message
print("📥 Відповідь моделі для DevOps запиту:")
print(response_message)

# Додаткові тестові запити
test_queries = [
    "Проаналізуй мої навички: Python, Git, Azure. Я розробник, хочу стати DevOps Engineer",
    "Створи план навчання DevOps на 6 місяців для початківця",
    "Які інструменти рекомендуєш для маленької команди розробників веб-додатків?",
    "Знайди курси підготовки до сертифікації AZ-400"
]

print("\n🎯 Інші приклади DevOps запитів:")
for query in test_queries:
    print(f"• {query}")

📥 Відповідь моделі для DevOps запиту:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"role":"developer","experience_level":"beginner","technology_focus":"CI/CD, Docker","content_type":"course"}', 'name': 'search_devops_courses'}, 'id': 'call_vRTAEtk9UN3TtyIlh1Ixz9yP', 'type': 'function'}]}

🎯 Інші приклади DevOps запитів:
• Проаналізуй мої навички: Python, Git, Azure. Я розробник, хочу стати DevOps Engineer
• Створи план навчання DevOps на 6 місяців для початківця
• Які інструменти рекомендуєш для маленької команди розробників веб-додатків?
• Знайди курси підготовки до сертифікації AZ-400
